In [19]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import count, col, desc
from pathlib import Path

# Start Spark
spark = SparkSession.builder \
    .appName("IPL-Pipeline") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("✅ Spark started")

# ── EXTRACT ──────────────────────────────────────────
print("\n[EXTRACT] Reading CSV...")
df = spark.read.csv(
    "data/matches.csv",
    header=True,
    inferSchema=True
)
print(f"  Rows loaded : {df.count()}")
print(f"  Columns     : {len(df.columns)}")

# ── TRANSFORM ────────────────────────────────────────
print("\n[TRANSFORM] Creating datasets...")

# 1. Team wins
team_wins = df.filter(col("winner").isNotNull()) \
    .groupBy("winner") \
    .agg(count("*").alias("total_wins")) \
    .orderBy(desc("total_wins"))

# 2. Season summary
season_summary = df.groupBy("season") \
    .agg(count("*").alias("matches_played")) \
    .orderBy("season")

# 3. Toss analysis
toss_analysis = df.filter(col("winner").isNotNull()) \
    .groupBy("toss_decision") \
    .agg(count("*").alias("total_matches")) \
    .orderBy(desc("total_matches"))

print("  ✅ team_wins created")
print("  ✅ season_summary created")
print("  ✅ toss_analysis created")

# ── LOAD to Parquet ───────────────────────────────────
print("\n[LOAD] Writing Parquet files...")

# Path("output").mkdir(exist_ok=True)

team_wins.write.mode("overwrite").parquet("output/team_wins")
season_summary.write.mode("overwrite").parquet("output/season_summary")
toss_analysis.write.mode("overwrite").parquet("output/toss_analysis")

print("  ✅ output/team_wins/")
print("  ✅ output/season_summary/")
print("  ✅ output/toss_analysis/")

# Quick preview
print("\n--- Top 5 Teams by Wins ---")
team_wins.show(5)

print("\n--- Matches Per Season ---")
season_summary.show(5)

✅ Spark started

[EXTRACT] Reading CSV...
  Rows loaded : 1095
  Columns     : 20

[TRANSFORM] Creating datasets...
  ✅ team_wins created
  ✅ season_summary created
  ✅ toss_analysis created

[LOAD] Writing Parquet files...
  ✅ output/team_wins/
  ✅ output/season_summary/
  ✅ output/toss_analysis/

--- Top 5 Teams by Wins ---
+--------------------+----------+
|              winner|total_wins|
+--------------------+----------+
|      Mumbai Indians|       144|
| Chennai Super Kings|       138|
|Kolkata Knight Ri...|       131|
|Royal Challengers...|       116|
|    Rajasthan Royals|       112|
+--------------------+----------+
only showing top 5 rows


--- Matches Per Season ---
+-------+--------------+
| season|matches_played|
+-------+--------------+
|2007/08|            58|
|   2009|            57|
|2009/10|            60|
|   2011|            73|
|   2012|            74|
+-------+--------------+
only showing top 5 rows



In [27]:
import boto3
import os
from dotenv import load_dotenv

load_dotenv("/home/jovyan/work/.env", override=True)

s3 = boto3.client(
    "s3",
    aws_access_key_id=os.getenv("AWS_ACCESS_KEY_ID"),
    aws_secret_access_key=os.getenv("AWS_SECRET_ACCESS_KEY"),
    region_name=os.getenv("AWS_REGION", "ap-south-1")
)

BUCKET_NAME = os.getenv("BUCKET_NAME")

# Test 1 — Can we talk to AWS at all?
print("Test 1: Connecting to AWS...")
try:
    buckets = s3.list_buckets()
    print(f"✅ Connected! Existing buckets: {[b['Name'] for b in buckets['Buckets']]}")
except Exception as e:
    print(f"❌ Connection failed: {e}")

# Test 2 — Try creating bucket with detailed error
print(f"\nTest 2: Creating bucket {BUCKET_NAME}...")
try:
    s3.create_bucket(
        Bucket=BUCKET_NAME,
        CreateBucketConfiguration={"LocationConstraint": "ap-south-1"}
    )
    print(f"✅ Bucket created successfully")
except Exception as e:
    print(f"❌ Error: {e}")

# Test 3 — Check if bucket exists now
print(f"\nTest 3: Checking if bucket exists...")
try:
    s3.head_bucket(Bucket=BUCKET_NAME)
    print(f"✅ Bucket exists and accessible")
except Exception as e:
    print(f"❌ Bucket check failed: {e}")


Test 1: Connecting to AWS...
✅ Connected! Existing buckets: ['ipl-pipeline-manish', 'manish-de-pipeline']

Test 2: Creating bucket ipl-pipeline-manish...
❌ Error: An error occurred (BucketAlreadyOwnedByYou) when calling the CreateBucket operation: Your previous request to create the named bucket succeeded and you already own it.

Test 3: Checking if bucket exists...
✅ Bucket exists and accessible


In [25]:
import boto3
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv("/home/jovyan/work/.env", override=True)

AWS_ACCESS_KEY = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_REGION     = os.getenv("AWS_REGION", "ap-south-1")
BUCKET_NAME    = os.getenv("BUCKET_NAME")

s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_ACCESS_KEY,
    aws_secret_access_key=AWS_SECRET_KEY,
    region_name=AWS_REGION
)

# ── STEP 1: Create bucket if it doesn't exist ────────
def create_bucket(bucket, region):
    try:
        s3.head_bucket(Bucket=bucket)
        print(f"✅ Bucket already exists: {bucket}")
    except:
        print(f"  Bucket not found. Creating {bucket} in {region}...")
        s3.create_bucket(
            Bucket=bucket,
            CreateBucketConfiguration={"LocationConstraint": region}
        )
        print(f"✅ Bucket created: {bucket}")

# ── STEP 2: Upload folder (skip .crc files) ──────────
def upload_folder_to_s3(local_dir, bucket, s3_prefix):
    local_path = Path(local_dir)

    if not local_path.exists():
        print(f"  ❌ Folder not found: {local_dir}")
        return False

    # Only upload real data files — skip Spark metadata files
    files = [
        f for f in local_path.iterdir()
        if f.is_file()
        and not f.name.startswith(".")    # skip .crc, ._SUCCESS
        and f.name != "_SUCCESS"          # skip _SUCCESS marker
        and f.name != "_committed"
    ]

    if not files:
        print(f"  ❌ No uploadable files in {local_dir}")
        return False

    print(f"\n  📁 {local_dir} → {len(files)} file(s)")
    for file_path in files:
        s3_key = f"{s3_prefix}{file_path.name}"
        print(f"     Uploading {file_path.name}...", end=" ")
        s3.upload_file(str(file_path), bucket, s3_key)
        print("✅")

    return True

# ── STEP 3: Verify what's in S3 ──────────────────────
def verify_upload(bucket):
    print(f"\n[VERIFY] Contents of s3://{bucket}/raw/")
    response = s3.list_objects_v2(Bucket=bucket, Prefix="raw/")

    if "Contents" not in response:
        print("  ❌ Nothing found in bucket")
        return

    total_kb = 0
    for obj in response["Contents"]:
        size_kb = round(obj["Size"] / 1024, 2)
        total_kb += size_kb
        print(f"  ✅ {obj['Key']}  ({size_kb} KB)")

    print(f"\n  Total: {round(total_kb, 2)} KB uploaded")

# ── RUN ──────────────────────────────────────────────
print("=" * 50)
print("DAY 3 — Create Bucket + Upload to S3")
print("=" * 50)

# 1. Create bucket
create_bucket(BUCKET_NAME, AWS_REGION)

# 2. Upload all 3 folders
folders = [
    ("output/team_wins",      "raw/team_wins/"),
    ("output/season_summary", "raw/season_summary/"),
    ("output/toss_analysis",  "raw/toss_analysis/"),
]

print("\n[UPLOAD] Starting...")
for local_dir, s3_prefix in folders:
    upload_folder_to_s3(local_dir, BUCKET_NAME, s3_prefix)

# 3. Verify
verify_upload(BUCKET_NAME)

print("\n🎉 Day 3 Complete!")
print(f"   Pipeline so far: CSV → PySpark → Parquet → S3")
print(f"   S3 bucket: s3://{BUCKET_NAME}/raw/")

DAY 3 — Create Bucket + Upload to S3
✅ Bucket already exists: ipl-pipeline-manish

[UPLOAD] Starting...

  📁 output/team_wins → 1 file(s)
     Uploading part-00000-7f57e33e-44de-4562-9440-9a0a454ccd28-c000.snappy.parquet... ✅

  📁 output/season_summary → 1 file(s)
     Uploading part-00000-bb35a441-4269-4081-88d0-efe09701447d-c000.snappy.parquet... ✅

  📁 output/toss_analysis → 1 file(s)
     Uploading part-00000-cf95f745-0dbc-4581-a124-3956d90f02a0-c000.snappy.parquet... ✅

[VERIFY] Contents of s3://ipl-pipeline-manish/raw/
  ✅ raw/season_summary/part-00000-bb35a441-4269-4081-88d0-efe09701447d-c000.snappy.parquet  (0.88 KB)
  ✅ raw/team_wins/part-00000-7f57e33e-44de-4562-9440-9a0a454ccd28-c000.snappy.parquet  (1.23 KB)
  ✅ raw/toss_analysis/part-00000-cf95f745-0dbc-4581-a124-3956d90f02a0-c000.snappy.parquet  (0.75 KB)

  Total: 2.86 KB uploaded

🎉 Day 3 Complete!
   Pipeline so far: CSV → PySpark → Parquet → S3
   S3 bucket: s3://ipl-pipeline-manish/raw/
